# Anomaly Detection (Isolation Forest)-Unsupervised

In [3]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

In [6]:
#load engineered features
video_features = pd.read_csv("data/processed/video_features_engineered.csv")

In [7]:
video_features.shape

(9999, 14)

In [10]:
# Select features for the model
feature_cols = [
    "like_view_ratio",
    "max_like_spike",
    "max_view_spike", 
    "like_spike_ratio",
    "view_spike_ratio",
    "peak_hour",
    "hours_tracked"
]

X = video_features[feature_cols].fillna(0)

# Scale features - important for Isolation Forest
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train Isolation Forest
# contamination = expected % of anomalies in data
# We estimate ~5% of videos have fake engagement
iso_forest = IsolationForest(contamination=0.08, random_state=42)
video_features["anomaly_score"] = iso_forest.fit_predict(X_scaled)
video_features["is_suspicious"] = video_features["anomaly_score"] == -1

print(f"Total videos analysed  : {len(video_features):,}")
print(f"Flagged as suspicious  : {video_features['is_suspicious'].sum():,}")
print(f"Suspicion rate         : {video_features['is_suspicious'].mean():.1%}")

Total videos analysed  : 9,999
Flagged as suspicious  : 800
Suspicion rate         : 8.0%


## Sustainability Feature

In [ ]:
### Measures whether engagement sustains AFTER the peak spike. Suspicious videos often have a big spike 
# followed by a sharp drop (dump & leave), while organic content has sustained engagement over time.
### Low sustainability = bot signature (dump & leave)
### High sustainability = organic viral content

In [11]:
yt_sample = pd.read_csv("data/processed/yt_sample.csv")

In [12]:
def add_sustainability(df_raw):
    results = []
    for video_id, group in df_raw.groupby("video_id"):
        group = group.sort_values("created_at")
        if len(group) < 3:
            continue
        like_growth = group["likes"].diff().fillna(0)
        # Find the hour of maximum spike
        spike_idx = like_growth.idxmax()
        spike_pos = group.index.get_loc(spike_idx)
        # Look at what happens AFTER the spike (next 3 hours)
        if spike_pos + 3 < len(group):
            after_spike = like_growth.iloc[spike_pos+1 : spike_pos+4].mean()
            spike_value = like_growth.iloc[spike_pos]
            # Ratio of post-spike activity to spike itself
            # Low ratio = dropped immediately = bot signature
            sustainability = after_spike / (spike_value + 1)
        else:
            sustainability = 0
        results.append({
        "video_id" : video_id,
        "sustainability" : sustainability
        })
    return pd.DataFrame(results)

sustainability_df = add_sustainability(yt_sample)

Calculating sustainability... (takes ~1 minute)


In [13]:
# Merge back into feature dataframe
video_features = video_features.merge(sustainability_df, on="video_id", how="left")
print(f"Sustainability stats:")
print(video_features["sustainability"].describe())

Sustainability stats:
count    9999.000000
mean        0.134525
std         0.216955
min        -0.619048
25%         0.000000
50%         0.000000
75%         0.238492
max         0.952381
Name: sustainability, dtype: float64


In [16]:
video_features.to_csv("data/processed/engineered_video_features2.csv", index=False)

In [14]:
# Compare suspicious vs normal
suspicious = video_features[video_features['is_suspicious']]
normal = video_features[~video_features['is_suspicious']]
print(f"Suspicious avg sustainability: {suspicious['sustainability'].mean():.3f}")
print(f"Normal avg sustainability : {normal['sustainability'].mean():.3f}")

Suspicious avg sustainability: 0.322
Normal avg sustainability : 0.118
